# Lecture 15 — Testing, documentation and continuous integration

We begin with an RK4 implementation containing a deliberate error. Its three existing tests pass.
We examine what they check, add a test that detects the error, and then organise a more complete suite.

The sections follow the [lecture notes](https://teaching.borisbolliet.io/c1-2026/materials/class15/notes.pdf).
The code is in [`attractor`](https://github.com/borisbolliet/attractor), at the tags
`lecture-15-start` and `lecture-15`.

## Setup

Use a Python 3.13 kernel. The tox example also requires Python 3.13 to be installed. The setup cell creates a temporary clone and a separate virtual
environment for the demonstrations. It installs the package with its test and fitting dependencies.
An internet connection is needed for downloads. The original repository and the notebook's Python
environment are not modified.

The terminal equivalent, after saving local changes, is:

```bash
cd ~/GitHub/attractor
git fetch origin --force tag lecture-15-start tag lecture-15
git switch --detach lecture-15-start
source setup_env.sh
```

In this notebook, `sh` runs a command in the temporary clone, using its virtual environment.
`py` runs a block of Python code there. Each call starts a new process, so an import uses the current
version of the files. A deliberately failing command is marked with `expect=1`; any unexpected exit
status stops the notebook.

In [1]:
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select a Python 3.13 kernel before running this notebook.")

remote = os.environ.get("ATTRACTOR_REMOTE", "https://github.com/borisbolliet/attractor.git")
if "ATTRACTOR_REMOTE" not in os.environ and Path("src/attractor").is_dir():
    probe = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
    if probe.returncode == 0:
        remote = probe.stdout.strip()

sandbox = Path(tempfile.mkdtemp(prefix="attractor-l15-"))
work = sandbox / "attractor"
subprocess.run(["git", "clone", "--quiet", "--no-hardlinks", "--no-checkout", remote, str(work)], check=True)
subprocess.run(["git", "-c", "advice.detachedHead=false", "switch", "--quiet", "--detach",
                "lecture-15-start"], cwd=work, check=True)
subprocess.run([sys.executable, "-m", "venv", str(sandbox / "venv")], check=True)
python = sandbox / "venv/bin/python"
env = dict(os.environ)
for name in ("PYTHONPATH", "VIRTUAL_ENV"):
    env.pop(name, None)
env.update(PATH=f"{python.parent}{os.pathsep}{env['PATH']}",
           PYTHONNOUSERSITE="1", PYTHONDONTWRITEBYTECODE="1",
           PIP_DISABLE_PIP_VERSION_CHECK="1", PYTHON_COLORS="1")


def run(args, *, cwd=None, expect=0, tail=None):
    result = subprocess.run(args, cwd=cwd or work, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = result.stdout.rstrip().splitlines()
    if tail and len(lines) > tail:
        print("[...]")
    print("\n".join(lines[-tail:] if tail else lines))
    if result.returncode != expect:
        raise RuntimeError(f"Expected exit status {expect}, got {result.returncode}: {args[0]}")
    return result


def sh(command, **kwargs):
    print("$", command)
    args = shlex.split(command)
    if args[0] == "python":
        args[0] = str(python)
    return run(args, **kwargs)


def py(source, **kwargs):
    return run([str(python), "-c", source], **kwargs)


def replace_once(path, old, new):
    text = path.read_text()
    if text.count(old) != 1:
        raise ValueError(f"Expected one occurrence of {old!r}")
    path.write_text(text.replace(old, new))
    for cache in (work / "src").rglob("__pycache__"):
        shutil.rmtree(cache)


_ = sh('python -m pip install -q -e ".[test,fit]"')
print("Demonstration clone:", work)

$ python -m pip install -q -e ".[test,fit]"



Demonstration clone: /var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/attractor


## 1 Tests with pytest

### A first test

An automated **test** runs code and checks an expected behaviour. For $dy/dt=2$, starting at
$y(0)=1$, a step to $t=0.5$ should give $y=2$. The first test checks this for Euler and RK4:

```python
def test_constant_rhs():
    assert euler_step(lambda t, y: 2.0, 0.0, 1.0, 0.5) == 2.0
    assert rk4_step(lambda t, y: 2.0, 0.0, 1.0, 0.5) == 2.0
```

A false `assert` condition raises `AssertionError`, making the test fail.
The file [`tests/test_integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-15-start/tests/test_integrators.py) also checks the returned array shapes
and the decay equation $dy/dt=-y$ at $t=1$.

pytest normally searches the current directory and its subdirectories for `test_*.py` and `*_test.py`,
subject to exclusions. Here, `testpaths = ["tests"]` in `pyproject.toml` restricts that search to `tests/`.
The option `-v` prints the result of each test.

In [2]:
_ = sh("python -m pytest -v --color=yes", tail=10)

$ python -m pytest -v --color=yes


[...]
configfile: pyproject.toml
testpaths: tests
plugins: cov-7.1.0
collecting ... collected 3 items

tests/test_integrators.py::test_constant_rhs PASSED                      [ 33%]
tests/test_integrators.py::test_shapes PASSED                            [ 66%]
tests/test_integrators.py::test_decay PASSED                             [100%]

============================== 3 passed in 0.26s ===============================


For the decay equation with $y(0)=1$, the exact value at $t=1$ is $e^{-1}$.
The test allows a relative difference of $10^{-3}$:

```python
def test_decay():
    t, y = integrate(rk4_step, lambda t, y: -y, [1.0], 0.0, 1.0, 0.1)
    assert math.isclose(y[-1, 0], math.exp(-1), rel_tol=1e-3)
```

All three assertions pass. This establishes the checks for these inputs, rather than correctness for every input.

### Statement coverage

We can also ask which statements ran. **Statement coverage** is the fraction of executable statements
reached by the tests. `pytest-cov` is a plugin that adds coverage measurement to pytest.
`--cov=attractor` selects the package; `--cov-report=term-missing` lists unexecuted lines.

In [3]:
_ = sh("python -m pytest --cov=attractor --cov-report=term-missing --color=yes", tail=16)

$ python -m pytest --cov=attractor --cov-report=term-missing --color=yes


[...]
================================ tests coverage ================================
______________ coverage: platform darwin, python 3.13.11-final-0 _______________

Name                           Stmts   Miss  Cover   Missing
------------------------------------------------------------
src/attractor/__init__.py          8      0   100%
src/attractor/cli.py              75     75     0%   2-105
src/attractor/cycles.py           18     15    17%   7-17, 22-25
src/attractor/datasets.py         11      7    36%   10-13, 18-20
src/attractor/integrators.py      35      8    77%   38, 41, 45-46, 53, 56, 60-61
src/attractor/models.py           56     14    75%   24, 28, 41-42, 57-59, 68-69, 72, 75, 78, 90, 96
src/attractor/tools.py            31     20    35%   8-18, 23-33, 49
src/attractor/trajectory.py       45     24    47%   19-20, 23, 27, 30, 34-36, 41-43, 47-50, 54-56, 60-63, 67, 71
------------------------------------------------------------
TOTAL                            279    1

The missing lines in `integrators.py` belong to methods of `Euler` and `RK4`.
All five statements in `rk4_step` ran: the four slope assignments and the return statement.
Coverage records their execution; it does not check their formulas.

### Branch coverage

An `if` condition can be true or false. Calling `absolute_value(-2)` below runs every statement,
but never checks the false outcome of `x < 0`:

```python
def absolute_value(x):
    if x < 0:
        x = -x
    return x
```

We put this small example in a separate directory inside the temporary clone.
The option `--cov-branch` also measures the possible branch outcomes.

In [4]:
examples = work / "examples"
examples.mkdir(exist_ok=True)
(examples / "branch_case.py").write_text(
    "def absolute_value(x):\n    if x < 0:\n        x = -x\n    return x\n"
)
branch_test = examples / "test_branch_case.py"
branch_test.write_text(
    "from branch_case import absolute_value\n\n"
    "def test_negative():\n    assert absolute_value(-2) == 2\n"
)
_ = sh("python -m pytest -q test_branch_case.py --cov=branch_case --cov-branch "
       "--cov-report=term-missing --color=yes", cwd=examples)

$ python -m pytest -q test_branch_case.py --cov=branch_case --cov-branch --cov-report=term-missing --color=yes
.                                                                        [100%]
================================ tests coverage ================================
______________ coverage: platform darwin, python 3.13.11-final-0 _______________

Name             Stmts   Miss Branch BrPart  Cover   Missing
------------------------------------------------------------
branch_case.py       4      0      2      1    83%   2->4
------------------------------------------------------------
TOTAL                4      0      2      1    83%
1 passed in 0.01s


Adding a positive input checks the other outcome. Both tests should pass, and both branches should now be covered.

In [5]:
with branch_test.open("a") as handle:
    handle.write("\ndef test_positive():\n    assert absolute_value(2) == 2\n")
_ = sh("python -m pytest -q test_branch_case.py --cov=branch_case --cov-branch "
       "--cov-report=term-missing --color=yes", cwd=examples)

$ python -m pytest -q test_branch_case.py --cov=branch_case --cov-branch --cov-report=term-missing --color=yes


..                                                                       [100%]
================================ tests coverage ================================
______________ coverage: platform darwin, python 3.13.11-final-0 _______________

Name             Stmts   Miss Branch BrPart  Cover   Missing
------------------------------------------------------------
branch_case.py       4      0      2      0   100%
------------------------------------------------------------
TOTAL                4      0      2      0   100%
2 passed in 0.02s


### An incorrect RK4 step

We have deliberately introduced an error in the third slope: `k3` uses `k1` where RK4 requires `k2`.
The diff shows the original line in red and its replacement in green.

In [6]:
_ = sh("git diff --color=always lecture-12 lecture-15-start -- src/attractor/integrators.py")

$ git diff --color=always lecture-12 lecture-15-start -- src/attractor/integrators.py
diff --git a/src/attractor/integrators.py b/src/attractor/integrators.py
index 5b0f3de..305dc49 100644
--- a/src/attractor/integrators.py
+++ b/src/attractor/integrators.py
@@ -14,10 +14,9 @@ def rk4_step(rhs, t, y, h):
     """One classical Runge-Kutta step of order four."""
     k1 = rhs(t, y)
     k2 = rhs(t + h / 2, y + h / 2 * k1)
-    k3 = rhs(t + h / 2, y + h / 2 * k2)
+    k3 = rhs(t + h / 2, y + h / 2 * k1)
     k4 = rhs(t + h, y + h * k3)
-    y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
-    return y
+    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
 
 
 def integrate(step, rhs, y0, t0, t1, h):


In [7]:
_ = py("""
import math
from attractor.integrators import integrate, rk4_step

t, y = integrate(rk4_step, lambda t, y: -y, [1.0], 0.0, 1.0, 0.1)
exact = math.exp(-1)
print(f"Computed: {y[-1, 0]:.6f}; exact: {exact:.6f}")
print(f"Relative error: {abs(y[-1, 0] - exact) / abs(exact):.2g}")
""")

Computed: 0.368202; exact: 0.367879
Relative error: 0.00088


The relative error is about $8.8\times10^{-4}$. It is below the test's tolerance of $10^{-3}$, so the assertion passes despite the mistake.

## 2 Floating-point comparison

### Relative and absolute tolerance

As in Lecture 2, the binary64 representation of `0.1` and `0.2` is rounded. Their sum need not
be exactly the stored value of `0.3`.

For a computed value $a$ and a reference $b$, a **relative tolerance** $r$ allows $r|b|$ of difference.
An **absolute tolerance** $\varepsilon$ allows a fixed difference. With both specified,
`pytest.approx` accepts

$$|a-b|\leq\max(r|b|,\varepsilon).$$

In [8]:
_ = py("""
import pytest
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2 == pytest.approx(0.3, rel=1e-6, abs=1e-12))
""")

0.30000000000000004
False
True


If $b=0$, the relative bound is zero, so we need an absolute tolerance to allow any non-zero difference.
For numerical integration there is also **discretisation error**, from using a finite step size.
A tolerance should reflect an accuracy requirement or an independent error estimate.

### Comparison functions

These functions combine tolerances differently. For finite $a,b$:

| Function | Allowed difference |
|---|---|
| `math.isclose` | $\max(r\max(|a|,|b|),\varepsilon)$ |
| `pytest.approx` (both bounds specified) | $\max(r|b|,\varepsilon)$ |
| `np.testing.assert_allclose` | $r|b|+\varepsilon$ |

For $a=1.15$, $b=1.0$, $r=0.1$ and $\varepsilon=0.1$, the allowed differences are 0.115, 0.1
and 0.2 respectively. The actual difference is 0.15.

In [9]:
_ = py("""
import math
import numpy as np
import pytest
print(math.isclose(1.15, 1.0, rel_tol=0.1, abs_tol=0.1))
print(1.15 == pytest.approx(1.0, rel=0.1, abs=0.1))
np.testing.assert_allclose([1.15], [1.0], rtol=0.1, atol=0.1)
""")

False
False


NumPy's assertion completes without output. If only `abs` is supplied to `pytest.approx`, that bound is used alone.

## 3 Testing a property

### Order of convergence

We can test how the error changes when we halve the step. An ordinary-differential-equation method
of order $p$ has global error $e(h)=O(h^p)$ at a fixed final time, for sufficiently smooth problems.
If the leading error is $e(h)\approx Ch^p$ with $C\ne0$, then

$$\frac{e(h)}{e(h/2)}\approx2^p,\qquad p_{\rm estimated}=\log_2\!\frac{e(h)}{e(h/2)}.$$

Classical RK4 has order four, so we expect an error ratio near sixteen. The decay equation provides
an exact reference and a measurable error; the constant-right-hand-side example is exact and cannot
be used for this ratio.

The file [`test_order.py`](https://github.com/borisbolliet/attractor/blob/lecture-15-start/test_order.py) is included at both lecture tags:

```python
import math
import pytest
from attractor.integrators import integrate, rk4_step


def error(h):
    t, y = integrate(rk4_step, lambda t, y: -y,
                     [1.0], 0.0, 1.0, h)
    return abs(y[-1, 0] - math.exp(-1))


def test_rk4_order():
    order = math.log2(error(0.1) / error(0.05))
    assert order == pytest.approx(4, abs=0.2)
```

We name it explicitly because it is outside the configured `tests/` directory.
The following failure is expected.

In [10]:
_ = sh("python -m pytest -q test_order.py --color=yes", expect=1)

$ python -m pytest -q test_order.py --color=yes


F                                                                        [100%]
=================================== FAILURES ===================================
________________________________ test_rk4_order ________________________________

    def test_rk4_order():
        order = math.log2(error(0.1) / error(0.05))
>       assert order == pytest.approx(4, abs=0.2)
E       assert 2.0362083816820737 == 4 ± 0.2
E         
E         comparison failed
E         Obtained: 2.0362083816820737
E         Expected: 4 ± 0.2

test_order.py:14: AssertionError
=========================== short test summary info ============================
FAILED test_order.py::test_rk4_order - assert 2.0362083816820737 == 4 ± 0.2
1 failed in 0.05s


The estimate is about 2.04, rather than four. We now replace `k1` with `k2` in the third slope
and rerun the same test.

In [11]:
integ = work / "src/attractor/integrators.py"
k3_correct = "k3 = rhs(t + h / 2, y + h / 2 * k2)"
k3_planted = "k3 = rhs(t + h / 2, y + h / 2 * k1)"
replace_once(integ, k3_planted, k3_correct)
_ = sh("python -m pytest -q test_order.py --color=yes")

$ python -m pytest -q test_order.py --color=yes


.                                                                        [100%]
1 passed in 0.03s


The expected order comes from the method. The unknown coefficient $C$ cancels in the error ratio,
so this test does not require an absolute error threshold for one step size.
We still need suitable steps and a tolerance for the estimated order: here $4\pm0.2$.
The steps must be small enough for the leading error to dominate, but not so small that rounding dominates.

### A conserved quantity and an equilibrium

When an exact solution is unavailable, a known property can still provide a test. For positive
populations $x,z$ in the Lotka–Volterra model,

$$\frac{dx}{dt}=ax-bxz,\qquad\frac{dz}{dt}=cxz-dz,$$

where $a,b,c,d$ are positive rate constants, the quantity

$$V(x,z)=cx-d\ln x+bz-a\ln z$$

is constant along an exact solution. At $(x,z)=(d/c,a/b)$, both derivatives vanish.
We first check this equilibrium directly, independently of the integrator.

In [12]:
_ = py("""
import numpy as np
from attractor import LotkaVolterra
m = LotkaVolterra()
derivatives = m.rhs(0.0, (m.d / m.c, m.a / m.b))
print(derivatives)
np.testing.assert_allclose(derivatives, 0.0, atol=1e-12)
""")

[0. 0.]


The comparison with zero uses an absolute tolerance. We can also measure the variation in $V$
along a numerical solution. `np.ptp` is the maximum minus the minimum:

In [13]:
_ = py("""
import numpy as np
from attractor import RK4, LotkaVolterra
m = LotkaVolterra()
tr = RK4(0.05).run(m, (1.5, 8.0), 0.0, 40.0)
x, z = tr.y[:, 0], tr.y[:, 1]
v = m.c*x - m.d*np.log(x) + m.b*z - m.a*np.log(z)
print(f"Spread relative to initial magnitude: {np.ptp(v) / abs(v[0]):.2g}")
assert np.ptp(v) < 1e-6 * abs(v[0])
""")

Spread relative to initial magnitude: 5.7e-08


## 4 Organising the tests

We now restore the completed code, tests and package configuration. Since `pyproject.toml` changes,
we also refresh the editable installation. In the terminal we can source `setup_env.sh` again;
here we use pip in the notebook's isolated environment.

The completed suite contains 20 tests and three doctests. The separate `test_order.py` remains
outside its configured search paths.

In [14]:
_ = sh("git restore --source lecture-15 -- src tests pyproject.toml")
_ = sh('python -m pip install -q -e ".[test,fit]"')
_ = sh("python -m pytest -q --color=yes")

$ git restore --source lecture-15 -- src tests pyproject.toml

$ python -m pip install -q -e ".[test,fit]"



$ python -m pytest -q --color=yes


.......................                                                  [100%]
23 passed in 6.83s


### Fixtures

Accuracy and convergence tests both need an equation and a reference value. A **fixture** prepares
these inputs in one place. pytest passes its return value to a test whose argument has the same name.
We can demonstrate this in a small test file:

In [15]:
fixture_example = '''
import math
import pytest
from attractor.integrators import integrate, rk4_step

@pytest.fixture
def problem():
    return (lambda t, y: -y), math.exp(-1)

def test_rk4_accuracy(problem):
    rhs, exact = problem
    t, y = integrate(rk4_step, rhs, [1.0], 0.0, 1.0, 0.1)
    assert y[-1, 0] == pytest.approx(exact, rel=1e-6)
'''
(examples / "test_fixture.py").write_text(fixture_example)
_ = sh("python -m pytest -v examples/test_fixture.py --color=yes", tail=6)

$ python -m pytest -v examples/test_fixture.py --color=yes
[...]
plugins: cov-7.1.0
collecting ... collected 1 item

examples/test_fixture.py::test_rk4_accuracy PASSED                       [100%]

============================== 1 passed in 0.03s ===============================


The test does not call `problem()` itself. pytest supplies the pair, which the test unpacks as
`rhs, exact`. A fixture shared by several files can go in `tests/conftest.py`.

### Parametrisation

**Parametrisation** repeats a test with specified inputs. The completed constant-right-hand-side test
uses one assertion for both methods:

```python
@pytest.mark.parametrize("step", [euler_step, rk4_step])
def test_constant_rhs(step):
    """dy/dt = 2 is integrated exactly by both methods."""
    assert step(lambda t, y: 2.0, 0.0, 1.0, 0.5) == 2.0
```

The fixture in [`tests/conftest.py`](https://github.com/borisbolliet/attractor/blob/lecture-15/tests/conftest.py) also supplies two equations:

```python
@pytest.fixture(params=["decay", "forced"])
def problem(request):
    """An equation dy/dt = rhs(t, y) with y(0) = 1, and its exact solution at t = 1.

    The fixture has two values, so every test that uses it runs twice.
    "forced" depends on t explicitly; "decay" does not.
    """
    if request.param == "decay":
        return (lambda t, y: -y), np.exp(-1.0)
    return (lambda t, y: np.cos(t) * y), np.exp(np.sin(1.0))
```

pytest provides each value through `request.param`. The second equation has exact solution
$y(t)=e^{\sin t}$ from $y(0)=1$. Two methods and two equations give four convergence tests.

In [16]:
_ = sh("python -m pytest -v tests/test_integrators.py -k convergence_order --color=yes", tail=9)

$ python -m pytest -v tests/test_integrators.py -k convergence_order --color=yes


[...]
plugins: cov-7.1.0
collecting ... collected 15 items / 11 deselected / 4 selected

tests/test_integrators.py::test_convergence_order[decay-euler_step-1] PASSED [ 25%]
tests/test_integrators.py::test_convergence_order[decay-rk4_step-4] PASSED [ 50%]
tests/test_integrators.py::test_convergence_order[forced-euler_step-1] PASSED [ 75%]
tests/test_integrators.py::test_convergence_order[forced-rk4_step-4] PASSED [100%]

======================= 4 passed, 11 deselected in 0.02s =======================


### Built-in exceptions and tracebacks

We have already seen `AssertionError` when an assertion is false. Other built-in exception types
report other problems:

| Exception | Example | Reason |
|---|---|---|
| `ValueError` | `float("abc")` | The string cannot represent a number. |
| `TypeError` | `len(3)` | An integer has no length. |
| `ZeroDivisionError` | `1 / 0` | Division by zero is undefined. |
| `IndexError` | `[10, 20][2]` | Only indices 0 and 1 exist. |
| `FileNotFoundError` | `open("missing.csv")` | The requested file does not exist. |

We can raise an exception explicitly. The next cell deliberately exits with an error.

In [17]:
_ = py('raise ValueError("step size h must be positive")', expect=1)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    raise ValueError("step size h must be positive")
ValueError: step size h must be positive


The **traceback** shows where the exception arose and the calls leading to it. The last line gives
the exception type and message. Here `<string>` refers to the code passed to `python -c` by `py`.

### Testing expected exceptions

Our example integrator requires equally sized steps that reach the endpoint. With $h=0.3$, its times
would be $0,0.3,0.6,0.9,1.2$, so it cannot reach 1. This is a design restriction of our implementation:
an RK4 solver can instead use a final step of 0.1, with 0.1 in every slope calculation for that step.

The completed code checks the inputs and raises `ValueError` with an explanatory message.

In [18]:
_ = py("""
from attractor.integrators import RK4, integrate, rk4_step
for operation in (
    lambda: RK4(-0.1),
    lambda: integrate(rk4_step, lambda t, y: -y, [1.0], 0.0, 1.0, 0.3),
):
    try:
        operation()
    except ValueError as error:
        print(type(error).__name__ + ":", error)
""")

ValueError: step size h must be positive, got -0.1
ValueError: (t1 - t0) / h = 3.33333 is not a whole number of steps


In a test suite, we want the test to **pass when the expected exception is raised**.
`pytest.raises` catches that exception. `match` checks the message against a regular expression:
here it must contain `must be positive`.

```python
@pytest.mark.parametrize("method", [Euler, RK4])
@pytest.mark.parametrize("h", [0.0, -0.1, float("nan")])
def test_step_must_be_positive(method, h):
    with pytest.raises(ValueError, match="must be positive"):
        method(h)
```

Two methods and three invalid step sizes give six runs. No exception, an unexpected type, or a
message without the required phrase makes a run fail.

In [19]:
_ = sh("python -m pytest -v tests/test_integrators.py -k step_must --color=yes", tail=12)

$ python -m pytest -v tests/test_integrators.py -k step_must --color=yes


[...]
plugins: cov-7.1.0
collecting ... collected 15 items / 8 deselected / 7 selected

tests/test_integrators.py::test_step_must_be_positive[0.0-Euler] PASSED  [ 14%]
tests/test_integrators.py::test_step_must_be_positive[0.0-RK4] PASSED    [ 28%]
tests/test_integrators.py::test_step_must_be_positive[-0.1-Euler] PASSED [ 42%]
tests/test_integrators.py::test_step_must_be_positive[-0.1-RK4] PASSED   [ 57%]
tests/test_integrators.py::test_step_must_be_positive[nan-Euler] PASSED  [ 71%]
tests/test_integrators.py::test_step_must_be_positive[nan-RK4] PASSED    [ 85%]
tests/test_integrators.py::test_step_must_divide_the_interval PASSED     [100%]

======================= 7 passed, 8 deselected in 0.02s ========================


The input check is `if not h > 0`. For `nan`, `h > 0` is false, so its negation rejects the input.
A check of `h <= 0` alone would not reject `nan`.

### Unit and integration tests

A **unit test** checks an individual component. An **integration test** checks components working
together. For example, `attractor-simulate` integrates a trajectory and estimates its period.
`capsys` is pytest's fixture for capturing printed output:

```python
def test_simulate(capsys):
    simulate(["--h", "0.05", "--days", "40"])
    out = capsys.readouterr().out
    m = re.match(r"801 points, step 0.05, algae period (\S+) days", out)
    assert m is not None
    assert float(m.group(1)) == pytest.approx(8.2, abs=0.1)
```

In [20]:
_ = sh("python -m pytest -v tests/test_cli.py --color=yes", tail=8)

$ python -m pytest -v tests/test_cli.py --color=yes


[...]
plugins: cov-7.1.0
collecting ... collected 3 items

tests/test_cli.py::test_simulate PASSED                                  [ 33%]
tests/test_cli.py::test_lynx_hare_is_installed PASSED                    [ 66%]
tests/test_cli.py::test_fit PASSED                                       [100%]

============================== 3 passed in 1.39s ===============================


We should also check that the package can read its datasets. An editable installation reads from
the project folder; that does not establish that the files are included in a wheel.

Some tests require optional dependencies. `test_fit` calls `pytest.importorskip("scipy")`, which skips
it if SciPy is absent. `python -m pytest -q -rs` includes the reasons for skipped tests.

### Checking a built wheel

To test the distributed package, we build a wheel and install it in a fresh environment. The data
check runs outside the source tree, using that environment's Python.

*Our example dataset has only 21 observations. Larger datasets would usually be downloaded when
requested and kept locally for reuse, rather than included in every wheel installation.*

In [21]:
wheelcheck = sandbox / "wheelcheck"
wheelcheck.mkdir()
_ = sh("python -m pip install -q build")
_ = sh(f"python -m build --wheel --outdir {shlex.quote(str(wheelcheck / 'dist'))}", tail=4)
_ = sh(f"python -m venv {shlex.quote(str(wheelcheck / 'venv'))}")
wheel_python = wheelcheck / "venv/bin/python"
wheel = next((wheelcheck / "dist").glob("*.whl"))
_ = run([str(wheel_python), "-m", "pip", "install", "-q", str(wheel)])
_ = run([str(wheel_python), "-c", """
from attractor import lynx_hare
year, lynx, hare = lynx_hare()
assert len(year) == len(lynx) == len(hare) == 21
assert year[0] == 1900 and year[-1] == 1920
"""], cwd=wheelcheck)

$ python -m pip install -q build



$ python -m build --wheel --outdir /var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/wheelcheck/dist


[...]
adding 'attractor-1.1.0.dist-info/top_level.txt'
adding 'attractor-1.1.0.dist-info/RECORD'
removing build/bdist.macosx-15.0-arm64/wheel
Successfully built attractor-1.1.0-py3-none-any.whl
$ python -m venv /var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/wheelcheck/venv


The last command prints nothing if the assertions pass. If the CSV file were missing from the wheel,
loading it would fail even if it existed in our development folder.

### Repeating the wheel test with tox

**tox** automates environment creation, wheel installation and test commands. This configuration
requests Python 3.13 and runs the existing tests against the installed wheel:

```toml
env_list = ["py313"]

[env_run_base]
package = "wheel"
extras = ["test", "fit"]
commands = [["python", "-m", "pytest", "-q", "tests/"]]
```

In [22]:
(work / "tox.toml").write_text('env_list = ["py313"]\n\n[env_run_base]\npackage = "wheel"\nextras = ["test", "fit"]\ncommands = [["python", "-m", "pytest", "-q", "tests/"]]\n')
_ = sh("python -m pip install -q tox")
_ = sh("python -m tox -e py313 -r", tail=16)

$ python -m pip install -q tox



$ python -m tox -e py313 -r


.pkg: remove tox env folder /private/var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/attractor/.tox/.pkg
.pkg: install_requires> python -I -m pip install 'setuptools>=77'
.pkg: _optional_hooks> python /private/var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/venv/lib/python3.13/site-packages/pyproject_api/_backend.py True setuptools.build_meta
.pkg: get_requires_for_build_wheel> python /private/var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/venv/lib/python3.13/site-packages/pyproject_api/_backend.py True setuptools.build_meta
.pkg: build_wheel> python /private/var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/venv/lib/python3.13/site-packages/pyproject_api/_backend.py True setuptools.build_meta
py313: install_package_deps> python -I -m pip install 'numpy>=1.26' 'pytest-cov>=6' 'pytest>=8' 'scipy>=1.11'
py313: install_package> python -I -m pip install --force-reinstall --no-deps /private/var/folder

The 20 tests in `tests/` run against the wheel. The three docstrings in `src/` are not collected by
this command. Our `src/` layout prevents imports from finding the development package at the
repository root. We can use the same tox command in CI after Python 3.13 is available.

## 5 Mutation testing

**Mutation testing** assesses the tests by deliberately changing the code. A changed version is a
**mutant**. A failing test kills it; if the tests all pass, it survives.

Starting from correct RK4, we change the time argument of the second slope:

```diff
- k2 = rhs(t + h / 2, y + h / 2 * k1)
+ k2 = rhs(t, y + h / 2 * k1)
```

For $dy/dt=-y$, the right-hand side ignores time, so this change has no effect. For $dy/dt=\cos(t)y$,
it changes the calculation. We run the same convergence test for each equation.

In [23]:
_ = sh("python -m pytest -q --color=yes")
original = integ.read_text()
try:
    replace_once(integ, "k2 = rhs(t + h / 2, y + h / 2 * k1)",
                 "k2 = rhs(t, y + h / 2 * k1)")
    _ = sh("python -m pytest -q tests/test_integrators.py -k 'convergence_order and decay' --color=yes")
    _ = sh("python -m pytest -q tests/test_integrators.py -k 'convergence_order and forced' --color=yes",
           expect=1)
finally:
    integ.write_text(original)

$ python -m pytest -q --color=yes


.......................                                                  [100%]
23 passed in 1.39s
$ python -m pytest -q tests/test_integrators.py -k 'convergence_order and decay' --color=yes
..                                                                       [100%]
2 passed, 13 deselected in 0.01s
$ python -m pytest -q tests/test_integrators.py -k 'convergence_order and forced' --color=yes


.F                                                                       [100%]
=================================== FAILURES ===================================
__________________ test_convergence_order[forced-rk4_step-4] ___________________

step = <function rk4_step at 0x106e74e00>, order = 4
problem = (<function problem.<locals>.<lambda> at 0x107364cc0>, np.float64(2.319776824715853))

    @pytest.mark.parametrize("step, order", [(euler_step, 1), (rk4_step, 4)])
    def test_convergence_order(step, order, problem):
        """Halving h divides the error by about 2**order."""
        ratio = error_at_1(step, problem, 0.1) / error_at_1(step, problem, 0.05)
>       assert math.log2(ratio) == pytest.approx(order, abs=0.2)
E       assert 0.9692333581520588 == 4 ± 0.2
E         
E         comparison failed
E         Obtained: 0.9692333581520588
E         Expected: 4 ± 0.2

tests/test_integrators.py:21: AssertionError
=========================== short test summary info ====================

The decay tests still pass. The RK4 order test for the time-dependent equation fails, detecting the changed
argument. This mutant is not equivalent to the original: there is an input for which the behaviour differs.

### Advanced testing with mutmut

[mutmut](https://mutmut.readthedocs.io/en/latest/) is an established tool for this optional technique.
Mutmut 3 uses pytest and creates one syntactic change per mutant, while leaving the tests unchanged.
Examples include replacing `h / 2` by `h * 2`, or `h > 0` by `h >= 0`.

The configuration in `pyproject.toml` selects the integrator module and its tests. To run the analysis
from a terminal in the completed project:

```bash
python -m pip install mutmut
mutmut run
mutmut results
```

Inspect a survivor with `mutmut show MUTANT_NAME`. It may reveal a missing test, or a change that
preserves the relevant behaviour. This analysis involves many test runs; the notebook demonstrates
one chosen mutation above rather than starting a full mutmut run.

## 6 Docstrings and documentation

### Docstrings in the NumPy convention

Our tests specify expected behaviour. Someone calling the function also needs to know its arguments,
return value and accepted inputs. A **docstring** is the first string literal in a function, class
or module; Python exposes it through `__doc__` and `help`.

The NumPy convention uses sections such as `Parameters`, `Returns`, `Raises` and `Examples`.
Here is the complete docstring of `euler_step` in [`src/attractor/integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-15/src/attractor/integrators.py):

```python
def euler_step(rhs, t, y, h):
    """One forward-Euler step, a method of order one.

    Parameters
    ----------
    rhs : callable
        Right-hand side ``rhs(t, y)`` of the equation dy/dt = rhs(t, y).
    t : float
        Time at the start of the step.
    y : float or numpy.ndarray
        State at time ``t``.
    h : float
        Step size.

    Returns
    -------
    float or numpy.ndarray
        Approximate state at time ``t + h``.

    Examples
    --------
    >>> euler_step(lambda t, y: -y, 0.0, 1.0, 0.1)
    0.9
    """
    return y + h * rhs(t, y)
```

In [24]:
_ = py("from attractor.integrators import euler_step\nhelp(euler_step)")

Help on function euler_step in module attractor.integrators:

euler_step(rhs, t, y, h)
    One forward-Euler step, a method of order one.

    Parameters
    ----------
    rhs : callable
        Right-hand side ``rhs(t, y)`` of the equation dy/dt = rhs(t, y).
    t : float
        Time at the start of the step.
    y : float or numpy.ndarray
        State at time ``t``.
    h : float
        Step size.

    Returns
    -------
    float or numpy.ndarray
        Approximate state at time ``t + h``.

    Examples
    --------
    >>> euler_step(lambda t, y: -y, 0.0, 1.0, 0.1)
    0.9


### Doctests

A **doctest** runs the commands marked `>>>` in a docstring and compares their printed output with
the expected text. The example is part of the function's documentation. Here is its location in
`rk4_step`; the parameter and return descriptions and numerical body are omitted:

```python
def rk4_step(rhs, t, y, h):
    """One classical Runge-Kutta step, a method of order four.

    [...]

    Examples
    --------
    >>> import math
    >>> y = rk4_step(lambda t, y: -y, 0.0, 1.0, 0.1)
    >>> print(f"{y:.8f}  {math.exp(-0.1):.8f}")
    0.90483750  0.90483742
    """
    ...
```

The first number is the RK4 result; the second is the exact value $e^{-0.1}$.
`.8f` prints eight digits after the decimal point. The import and assignment produce no output.
Doctest compares the whole printed line with the recorded text; it does not compare the two numbers
with each other.

`--doctest-modules` enables this check. At `lecture-15`, `pyproject.toml` already supplies that option.

In [25]:
_ = sh("python -m pytest -v --doctest-modules src/attractor/integrators.py --color=yes", tail=8)

$ python -m pytest -v --doctest-modules src/attractor/integrators.py --color=yes


[...]
plugins: platformdirs-4.14.5, cov-7.1.0
collecting ... collected 3 items

src/attractor/integrators.py::attractor.integrators.check_step PASSED    [ 33%]
src/attractor/integrators.py::attractor.integrators.euler_step PASSED    [ 66%]
src/attractor/integrators.py::attractor.integrators.rk4_step PASSED      [100%]

============================== 3 passed in 0.04s ===============================


Putting the original third-slope error back changes the printed RK4 value. The doctest should now fail; the file is restored afterwards.

In [26]:
original = integ.read_text()
try:
    replace_once(integ, k3_correct, k3_planted)
    _ = sh("python -m pytest -q --doctest-modules src/attractor/integrators.py --color=yes", expect=1)
finally:
    integ.write_text(original)

$ python -m pytest -q --doctest-modules src/attractor/integrators.py --color=yes


..F                                                                      [100%]
=================================== FAILURES ===================================
___________________ [doctest] attractor.integrators.rk4_step ___________________
054 float or numpy.ndarray
055     Approximate state at time ``t + h``.
056 
057 Examples
058 --------
059 One step of dy/dt = -y from y = 1, against the exact value exp(-0.1):
060 
061 >>> import math
062 >>> y = rk4_step(lambda t, y: -y, 0.0, 1.0, 0.1)
063 >>> print(f"{y:.8f}  {math.exp(-0.1):.8f}")
Expected:
    0.90483750  0.90483742
Got:
    0.90491667  0.90483742

/private/var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/attractor/src/attractor/integrators.py:63: DocTestFailure
=========================== short test summary info ============================
FAILED src/attractor/integrators.py::attractor.integrators.rk4_step
1 failed, 2 passed in 0.03s


An expected exception in a doctest is written as the first traceback line, `...`, and the final exception line:

```python
>>> check_step(0.3, 0.0, 1.0)
Traceback (most recent call last):
...
ValueError: (t1 - t0) / h = 3.33333 is not a whole number of steps
```

### A documentation site with Zensical

[Zensical](https://zensical.org/docs/) builds a website from Markdown pages.
The [mkdocstrings Python handler](https://mkdocstrings.github.io/python/usage/) reads our source code
and inserts the functions, classes and docstrings into those pages.

[`zensical.toml`](https://github.com/borisbolliet/attractor/blob/lecture-15/zensical.toml) names the pages and configures the handler:

```toml
# Zensical configuration: the documentation site, built from docs/ and the docstrings.
[project]
site_name = "attractor"
site_url = "https://borisbolliet.github.io/attractor/"
repo_url = "https://github.com/borisbolliet/attractor"
nav = [{ "Home" = "index.md" }, { "API reference" = "api.md" }]

# mkdocstrings reads the docstrings of the package in src/
[project.plugins.mkdocstrings.handlers.python]
paths = ["src"]
options = { docstring_style = "numpy" }
```

`nav` names pages relative to `docs/`; `paths = ["src"]` locates our modules. `docstring_style = "numpy"`
selects the convention used in their docstrings. In [`docs/api.md`](https://github.com/borisbolliet/attractor/blob/lecture-15/docs/api.md), this directive inserts
reference documentation for the integrators:

```markdown
# API reference

::: attractor.integrators
```

In [27]:
_ = sh("git restore --source lecture-15 -- docs zensical.toml .github")
_ = sh('python -m pip install -q -e ".[docs]"')
_ = sh("zensical build --strict")
assert (work / "site/api/index.html").is_file()
print("Built site:", work / "site")

$ git restore --source lecture-15 -- docs zensical.toml .github

$ python -m pip install -q -e ".[docs]"



$ zensical build --strict


Build started
No issues found
Build finished in 0.34s
Built site: /var/folders/hc/d14d998d3rv6_7vdcjskjrxc0000gp/T/attractor-l15-zl86uj85/attractor/site


`--strict` makes warnings fail the build. The website files are written to `site/`.
To preview them from a terminal in this clone, run `zensical serve` and open <http://localhost:8000/>.
`Ctrl+C` stops the server.

## 7 Continuous integration

### Workflows, jobs, steps and runners

We can run the tests automatically whenever changes are pushed or proposed in a pull request.
**Continuous integration** combines frequent integration of changes with automated builds and tests.

[GitHub Actions](https://docs.github.com/en/actions) runs a **workflow** defined by a YAML file in
`.github/workflows/`. A workflow contains **jobs**, each consisting of ordered **steps**.
A **runner** is the machine executing a job. Here GitHub supplies a fresh Ubuntu virtual machine.
The steps of that job use the same machine, so later steps can use the files and packages installed earlier.

A step uses `run` for a shell command or `uses` to invoke an **action**, reusable code for a task.
We begin with a complete example for one Python environment:

```yaml
on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.13"
      - run: python -m pip install -e ".[test]"
      - run: |
          python -m pytest -v --cov=attractor --cov-branch \
            --cov-report=term-missing
```

`on` specifies pushes to `main` and pull requests; `runs-on` selects the runner.
The four steps obtain the repository files, select Python, install the package with test dependencies,
and run pytest with coverage. `with` supplies the Python version to the setup action.

A failing test makes pytest return status 1. By default, that fails the step and job, and later steps
of the job are skipped. Installing the package on the runner can also expose an undeclared dependency
that happened to be available on our laptop.

### A test matrix

To check several environments, a **matrix** gives the values to combine. The completed workflow
uses two Python versions and two choices of dependencies. `${{ matrix.python }}` and
`${{ matrix.extras }}` are replaced by the selected values in each run.

```yaml
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python: ["3.13", "3.14"]
        extras: ["test", "test,fit"]
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: ${{ matrix.python }}
      - run: python -m pip install -e ".[${{ matrix.extras }}]"
      - run: python -m pytest -v --cov=attractor --cov-branch --cov-report=term-missing
```

In [28]:
import itertools
for version, extras in itertools.product(["3.13", "3.14"], ["test", "test,fit"]):
    print(f"Python {version}: python -m pip install -e '.[{extras}]'")

Python 3.13: python -m pip install -e '.[test]'
Python 3.13: python -m pip install -e '.[test,fit]'
Python 3.14: python -m pip install -e '.[test]'
Python 3.14: python -m pip install -e '.[test,fit]'


There are four separate jobs, which can run in parallel. `"test,fit"` is one string selecting both
extras, including SciPy. Version strings are quoted to prevent YAML interpreting them as numbers.
`requires-python = ">=3.13"` specifies a minimum; this matrix tests only 3.13 and 3.14.

### Dependent jobs and publication

The documentation job uses `needs: test` to wait for all matrix runs to succeed.
`if: github.event_name == 'push'` limits publication to pushes. The workflow's push filter restricts
these to `main`. On a pull request, only the tests run.

```yaml
docs:
    needs: test
    if: github.event_name == 'push'
    runs-on: ubuntu-latest
    permissions:
      pages: write
      id-token: write
    environment:
      name: github-pages
      url: ${{ steps.deploy.outputs.page_url }}
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.13"
      - run: python -m pip install -e ".[docs]"
      - run: zensical build --strict
      - uses: actions/upload-pages-artifact@v5
        with:
          path: site
      - id: deploy
        uses: actions/deploy-pages@v5
```

After `zensical build --strict`, `upload-pages-artifact` uploads the `site/` directory and
`deploy-pages` publishes it. This configuration describes remote jobs; running this notebook does
not dispatch a workflow or publish a website.

We can inspect a real run in the repository's Actions tab, or from a terminal authenticated with GitHub:

```bash
gh run list --limit 3
gh run view --log-failed
```

The first command lists recent runs. The second shows logs from failed steps of a selected run.
The notes contain the exercises and solutions.